In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

In [2]:
books_features = pd.read_csv("../Data/books_features.csv")
ratings = pd.read_csv("../Data/ratings.csv")

books_features.head()
ratings.head()

,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3


In [3]:
ratings["rating"].value_counts().sort_index()

rating
1     124195
2     359257
3    1370916
4    2139018
5    1983093
Name: count, dtype: int64

In [4]:
relevant_ratings = ratings[ratings["rating"] >= 4].copy()

relevant_ratings.head()

,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
5,2,26,4


In [5]:
relevant_ratings.shape

(4122111, 3)

In [6]:
relevant_ratings["user_id"].nunique(), relevant_ratings["book_id"].nunique()

(53406, 10000)

In [7]:
relevant_counts = (
    relevant_ratings
    .groupby("user_id")
    .size()
)

relevant_counts.describe()

count    53406.000000
mean        77.184417
std         26.038128
min          1.000000
25%         60.000000
50%         76.000000
75%         93.000000
max        194.000000
dtype: float64

In [8]:
eligible_users = relevant_counts[relevant_counts >= 2]

eligible_users.shape

(53398,)

In [9]:
eligible_users.describe()

count    53398.000000
mean        77.195831
std         26.023374
min          2.000000
25%         60.000000
50%         76.000000
75%         93.000000
max        194.000000
dtype: float64

In [10]:
np.random.seed(42)

user_book_groups = (
    relevant_ratings
    .groupby("user_id")["book_id"]
    .apply(np.array)
)

evaluation_pairs = []

for user_id, user_books in user_book_groups.items():
    
    if len(user_books) < 2:
        continue
    
    test_book = np.random.choice(user_books)
    
    remaining_books = user_books[user_books != test_book]
    seed_book = np.random.choice(remaining_books)
    
    evaluation_pairs.append(
        [user_id, seed_book, test_book]
    )

evaluation_pairs = pd.DataFrame(
    evaluation_pairs,
    columns=["user_id", "seed_book_id", "test_book_id"]
)

evaluation_pairs.head()

,user_id,seed_book_id,test_book_id
0,1,6285,162
1,2,127,8855
2,3,4090,7401
3,4,3469,10
4,5,852,262


In [11]:
print("Seed books found:",
      evaluation_pairs["seed_book_id"].isin(
          books_features["book_id"]
      ).sum())

print("Test books found:",
      evaluation_pairs["test_book_id"].isin(
          books_features["book_id"]
      ).sum())

print("Total evaluation pairs:",
      len(evaluation_pairs))

Seed books found: 53398
Test books found: 53398
Total evaluation pairs: 53398


In [12]:
evaluation_sample = evaluation_pairs.sample(
    n=1000,
    random_state=42
).reset_index(drop=True)

evaluation_sample.head()

,user_id,seed_book_id,test_book_id
0,2976,856,1015
1,14501,144,6406
2,52153,4593,466
3,30203,3492,2086
4,25134,3504,1552


In [13]:
book_id_to_index = pd.Series(
    books_features.index,
    index=books_features["book_id"]
).to_dict()

len(book_id_to_index)

10000

In [14]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(
    books_features["tag_name"]
)

content_similarity = cosine_similarity(tfidf_matrix)

content_similarity.shape

(10000, 10000)

In [15]:
from scipy.sparse import csr_matrix

user_ids = ratings["user_id"].astype("category")
book_ids = ratings["book_id"].astype("category")

user_item_matrix = csr_matrix(
    (
        ratings["rating"].values,
        (user_ids.cat.codes, book_ids.cat.codes)
    )
)

user_item_matrix.shape

(53424, 10000)

In [16]:
collaborative_model = NearestNeighbors(
    n_neighbors=11,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

collaborative_model.fit(user_item_matrix.T)

print("Collaborative model fitted.")

Collaborative model fitted.


In [17]:
evaluation_ratings = ratings.merge(
    evaluation_sample[
        ["user_id", "test_book_id"]
    ],
    on="user_id",
    how="left"
)

evaluation_ratings = evaluation_ratings[
    ~(
        (evaluation_ratings["book_id"] == evaluation_ratings["test_book_id"])
        & evaluation_ratings["test_book_id"].notna()
    )
].copy()

evaluation_ratings = evaluation_ratings[
    ["user_id", "book_id", "rating"]
]

evaluation_ratings.shape

(5975479, 3)

In [18]:
eval_user_ids = evaluation_ratings["user_id"].astype("category")
eval_book_ids = evaluation_ratings["book_id"].astype("category")

eval_user_item_matrix = csr_matrix(
    (
        evaluation_ratings["rating"].values,
        (eval_user_ids.cat.codes, eval_book_ids.cat.codes)
    ),
    shape=(len(eval_user_ids.cat.categories), len(eval_book_ids.cat.categories))
)

eval_user_item_matrix.shape

(53424, 10000)

In [19]:
eval_collaborative_model = NearestNeighbors(
    n_neighbors=11,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

eval_collaborative_model.fit(
    eval_user_item_matrix.T
)

print("Evaluation collaborative model fitted.")

Evaluation collaborative model fitted.


In [20]:
test_case = evaluation_sample.iloc[0]

test_case

user_id         2976
seed_book_id     856
test_book_id    1015
Name: 0, dtype: int64

In [21]:
seed_book_id = test_case["seed_book_id"]
test_book_id = test_case["test_book_id"]

seed_book_id, test_book_id

(np.int64(856), np.int64(1015))

In [22]:
seed_book = books_features[
    books_features["book_id"] == seed_book_id
][["book_id", "title", "authors"]]

test_book = books_features[
    books_features["book_id"] == test_book_id
][["book_id", "title", "authors"]]

print("SEED BOOK")
display(seed_book)

print("\nTEST BOOK")
display(test_book)

SEED BOOK


,book_id,title,authors
855,856,"Gathering Blue (The Giver, #2)",Lois Lowry



TEST BOOK


,book_id,title,authors
1014,1015,Steppenwolf,"Hermann Hesse, Basil Creighton"


In [23]:
seed_title = books_features.loc[
    books_features["book_id"] == seed_book_id,
    "title"
].iloc[0]

seed_title

'Gathering Blue (The Giver, #2)'

In [24]:
def recommend_content_eval(book_id, n=10):
    
    # Find the book's position
    if book_id not in book_id_to_index:
        return pd.DataFrame()
    
    idx = book_id_to_index[book_id]
    
    # Get similarity scores
    scores = content_similarity[idx]
    
    # Sort by similarity
    top_indices = np.argsort(scores)[::-1]
    
    # Remove the original book
    top_indices = [
        i for i in top_indices
        if i != idx
    ][:n]
    
    recommendations = books_features.iloc[top_indices].copy()
    
    recommendations["similarity"] = scores[top_indices]
    
    return recommendations[
        ["book_id", "title", "authors", "similarity"]
    ]

In [25]:
content_recommendations = recommend_content_eval(
    seed_book_id,
    n=10
)

content_recommendations

,book_id,title,authors,similarity
1289,1290,"Messenger (The Giver, #3)",Lois Lowry,0.863327
485,486,"The City of Ember (Book of Ember, #1)",Jeanne DuPrau,0.791112
1902,1903,"Son (The Giver, #4)",Lois Lowry,0.789708
6163,6164,"Among the Betrayed (Shadow Children, #3)",Margaret Peterson Haddix,0.784124
5540,5541,"Among the Impostors (Shadow Children, #2)",Margaret Peterson Haddix,0.771667
8509,8510,"Among the Enemy (Shadow Children, #6)",Margaret Peterson Haddix,0.765565
3083,3084,"The People of Sparks (Book of Ember, #2)",Jeanne DuPrau,0.764836
7303,7304,"Among the Barons (Shadow Children, #4)",Margaret Peterson Haddix,0.763559
5889,5890,"The Prophet of Yonwood (Book of Ember, #3)",Jeanne DuPrau,0.757566
6900,6901,"The Diamond of Darkhold (Book of Ember, #4)",Jeanne DuPrau,0.742242


In [26]:
test_book_id in content_recommendations["book_id"].values

False

In [27]:
content_matches = content_recommendations[
    content_recommendations["book_id"] == test_book_id
]

content_matches

,book_id,title,authors,similarity


In [28]:
def get_content_recommendations(book_id, k=10):
    
    if book_id not in book_id_to_index:
        return []
    
    idx = book_id_to_index[book_id]
    
    scores = content_similarity[idx]
    
    top_indices = np.argsort(scores)[::-1]
    
    # Remove the seed book
    top_indices = top_indices[top_indices != idx][:k]
    
    return books_features.iloc[top_indices]["book_id"].tolist()

In [29]:
content_test_recommendations = get_content_recommendations(
    seed_book_id,
    k=10
)

content_test_recommendations

[1290, 486, 1903, 6164, 5541, 8510, 3084, 7304, 5890, 6901]

In [30]:
test_book_id in content_test_recommendations

False

In [31]:
def get_collaborative_recommendations(book_id, k=10):
    
    if book_id not in eval_book_ids.cat.categories:
        return []
    
    book_index = eval_book_ids.cat.categories.get_loc(book_id)
    
    distances, indices = eval_collaborative_model.kneighbors(
        eval_user_item_matrix.T[book_index],
        n_neighbors=k + 1
    )
    
    # Convert cosine distance to similarity
    similarities = 1 - distances[0]
    
    # Remove the seed book
    similar_indices = indices[0][1:]
    
    similar_book_ids = eval_book_ids.cat.categories[
        similar_indices
    ]
    
    return similar_book_ids.tolist()

In [32]:
collaborative_test_recommendations = get_collaborative_recommendations(
    seed_book_id,
    k=10
)

collaborative_test_recommendations

[1290, 1903, 36, 241, 17, 20, 69, 91, 12, 259]

In [33]:
test_book_id in collaborative_test_recommendations

False

In [34]:
def get_hybrid_recommendations(book_id, alpha=0.5, k=10):
    
    if book_id not in book_id_to_index:
        return []
    
    content_idx = book_id_to_index[book_id]
    
    # -----------------------------
    # CONTENT SCORES
    # -----------------------------
    
    content_scores = content_similarity[content_idx].copy()
    
    # -----------------------------
    # COLLABORATIVE SCORES
    # -----------------------------
    
    collaborative_scores = np.zeros(len(books_features))
    
    if book_id in eval_book_ids.cat.categories:
        
        book_index = eval_book_ids.cat.categories.get_loc(book_id)
        
        distances, indices = eval_collaborative_model.kneighbors(
            eval_user_item_matrix.T[book_index],
            n_neighbors=k + 1
        )
        
        similar_indices = indices[0][1:]
        similar_scores = 1 - distances[0][1:]
        
        similar_book_ids = eval_book_ids.cat.categories[
            similar_indices
        ]
        
        for similar_book_id, score in zip(
            similar_book_ids,
            similar_scores
        ):
            
            if similar_book_id in book_id_to_index:
                row = book_id_to_index[similar_book_id]
                collaborative_scores[row] = score
    
    # -----------------------------
    # HYBRID SCORE
    # -----------------------------
    
    hybrid_scores = (
        alpha * content_scores
        + (1 - alpha) * collaborative_scores
    )
    
    # Remove seed book
    hybrid_scores[content_idx] = -1
    
    # Top-K
    top_indices = np.argsort(
        hybrid_scores
    )[::-1][:k]
    
    return books_features.iloc[
        top_indices
    ]["book_id"].tolist()

In [35]:
hybrid_test_recommendations = get_hybrid_recommendations(
    seed_book_id,
    alpha=0.5,
    k=10
)

hybrid_test_recommendations

[1290, 1903, 36, 486, 6164, 5541, 8510, 3084, 7304, 5890]

In [36]:
test_book_id in hybrid_test_recommendations

False

In [37]:
def evaluate_content(evaluation_data, k=10):
    
    hits = 0
    
    for _, row in evaluation_data.iterrows():
        
        recommendations = get_content_recommendations(
            row["seed_book_id"],
            k=k
        )
        
        if row["test_book_id"] in recommendations:
            hits += 1
    
    return hits / len(evaluation_data)

In [38]:
content_hit_rate_10 = evaluate_content(
    evaluation_sample,
    k=10
)

content_hit_rate_10

0.025

In [39]:
def evaluate_collaborative(evaluation_data, k=10):
    
    hits = 0
    
    for _, row in evaluation_data.iterrows():
        
        recommendations = get_collaborative_recommendations(
            row["seed_book_id"],
            k=k
        )
        
        if row["test_book_id"] in recommendations:
            hits += 1
    
    return hits / len(evaluation_data)

In [40]:
unique_seed_books = evaluation_sample["seed_book_id"].nunique()

print("Evaluation cases:", len(evaluation_sample))
print("Unique seed books:", unique_seed_books)

Evaluation cases: 1000
Unique seed books: 813


In [41]:
unique_seed_books = evaluation_sample["seed_book_id"].unique()

seed_indices = [
    eval_book_ids.cat.categories.get_loc(book_id)
    for book_id in unique_seed_books
]

seed_vectors = eval_user_item_matrix.T[seed_indices]

distances, indices = eval_collaborative_model.kneighbors(
    seed_vectors,
    n_neighbors=11
)

print("Queries processed:", len(unique_seed_books))
print("Distance matrix shape:", distances.shape)
print("Index matrix shape:", indices.shape)

Queries processed: 813
Distance matrix shape: (813, 11)
Index matrix shape: (813, 11)


In [42]:
collaborative_recommendation_map = {}

for i, book_id in enumerate(unique_seed_books):
    
    neighbor_indices = indices[i][1:]
    
    neighbor_book_ids = eval_book_ids.cat.categories[
        neighbor_indices
    ].tolist()
    
    collaborative_recommendation_map[book_id] = neighbor_book_ids

In [43]:
collaborative_recommendation_map[seed_book_id]

[1290, 1903, 36, 241, 17, 20, 69, 91, 12, 259]

In [44]:
collaborative_hits = 0

for _, row in evaluation_sample.iterrows():
    
    recommendations = collaborative_recommendation_map[
        row["seed_book_id"]
    ]
    
    if row["test_book_id"] in recommendations:
        collaborative_hits += 1

collaborative_hit_rate_10 = (
    collaborative_hits / len(evaluation_sample)
)

collaborative_hit_rate_10

0.043

In [45]:
hybrid_recommendation_map = {}

for i, book_id in enumerate(unique_seed_books):
    
    content_idx = book_id_to_index[book_id]
    
    # Content scores for this seed book
    content_scores = content_similarity[content_idx].copy()
    
    # Collaborative scores
    collaborative_scores = np.zeros(len(books_features))
    
    neighbor_indices = indices[i][1:]
    neighbor_scores = 1 - distances[i][1:]
    
    neighbor_book_ids = eval_book_ids.cat.categories[
        neighbor_indices
    ]
    
    for neighbor_book_id, score in zip(
        neighbor_book_ids,
        neighbor_scores
    ):
        
        if neighbor_book_id in book_id_to_index:
            row_index = book_id_to_index[neighbor_book_id]
            collaborative_scores[row_index] = score
    
    # Hybrid α = 0.5
    hybrid_scores = (
        0.5 * content_scores
        + 0.5 * collaborative_scores
    )
    
    # Remove seed book
    hybrid_scores[content_idx] = -1
    
    # Top 10
    top_indices = np.argsort(
        hybrid_scores
    )[::-1][:10]
    
    hybrid_recommendation_map[book_id] = (
        books_features.iloc[top_indices]["book_id"].tolist()
    )

print("Hybrid recommendations generated.")

Hybrid recommendations generated.


In [46]:
hybrid_recommendation_map[seed_book_id]

[1290, 1903, 36, 486, 6164, 5541, 8510, 3084, 7304, 5890]

In [47]:
hybrid_hits = 0

for _, row in evaluation_sample.iterrows():
    
    recommendations = hybrid_recommendation_map[
        row["seed_book_id"]
    ]
    
    if row["test_book_id"] in recommendations:
        hybrid_hits += 1

hybrid_hit_rate_10 = (
    hybrid_hits / len(evaluation_sample)
)

hybrid_hit_rate_10

0.039

In [48]:
hybrid_experiments = pd.read_csv(
    "../experiments/hybrid_experiments.csv"
)

hybrid_experiments

,experiment,content_weight,collaborative_weight,precision_at_10,recall_at_10,hit_rate_at_10,notes
0,EXP-01,0.5,0.5,NaN,NaN,NaN,Baseline -> 3.9% Hit@10
1,EXP-02,0.6,0.4,NaN,NaN,NaN,Current configuration
2,EXP-03,0.7,0.3,NaN,NaN,NaN,Content-heavy
3,EXP-04,0.8,0.2,NaN,NaN,NaN,Strong content
4,EXP-05,0.4,0.6,NaN,NaN,NaN,Collaborative-heavy
5,EXP-06,0.3,0.7,NaN,NaN,NaN,Strong collaborative


In [49]:
alpha_values = [0.5, 0.6, 0.7, 0.8, 0.4, 0.3]

hybrid_results = []

for alpha in alpha_values:
    
    hits = 0
    
    for _, row in evaluation_sample.iterrows():
        
        seed_book = row["seed_book_id"]
        test_book = row["test_book_id"]
        
        # Get the precomputed hybrid recommendations
        content_idx = book_id_to_index[seed_book]
        
        # Content scores
        content_scores = content_similarity[content_idx]
        
        # Find collaborative neighbors for this seed
        seed_position = np.where(
            unique_seed_books == seed_book
        )[0][0]
        
        neighbor_indices = indices[seed_position][1:]
        neighbor_scores = 1 - distances[seed_position][1:]
        
        collaborative_scores = np.zeros(len(books_features))
        
        neighbor_book_ids = eval_book_ids.cat.categories[
            neighbor_indices
        ]
        
        for neighbor_book_id, score in zip(
            neighbor_book_ids,
            neighbor_scores
        ):
            
            if neighbor_book_id in book_id_to_index:
                row_index = book_id_to_index[neighbor_book_id]
                collaborative_scores[row_index] = score
        
        # Hybrid
        hybrid_scores = (
            alpha * content_scores
            + (1 - alpha) * collaborative_scores
        )
        
        hybrid_scores[content_idx] = -1
        
        top_indices = np.argsort(
            hybrid_scores
        )[::-1][:10]
        
        recommended_ids = books_features.iloc[
            top_indices
        ]["book_id"].values
        
        if test_book in recommended_ids:
            hits += 1
    
    hit_rate = hits / len(evaluation_sample)
    
    hybrid_results.append({
        "alpha": alpha,
        "content_weight": alpha,
        "collaborative_weight": 1 - alpha,
        "hit_rate_at_10": hit_rate
    })

hybrid_results_df = pd.DataFrame(hybrid_results)

hybrid_results_df

,alpha,content_weight,collaborative_weight,hit_rate_at_10
0,0.5,0.5,0.5,0.039
1,0.6,0.6,0.4,0.036
2,0.7,0.7,0.3,0.032
3,0.8,0.8,0.2,0.029
4,0.4,0.4,0.6,0.040
5,0.3,0.3,0.7,0.042


In [50]:
experiment_metrics = []

for _, result in hybrid_results_df.iterrows():
    
    alpha = result["alpha"]
    
    hits = 0
    
    for _, row in evaluation_sample.iterrows():
        
        seed_book = row["seed_book_id"]
        test_book = row["test_book_id"]
        
        content_idx = book_id_to_index[seed_book]
        
        # Content scores
        content_scores = content_similarity[content_idx]
        
        # Locate this seed in our precomputed collaborative results
        seed_position = np.where(
            unique_seed_books == seed_book
        )[0][0]
        
        neighbor_indices = indices[seed_position][1:]
        neighbor_scores = 1 - distances[seed_position][1:]
        
        collaborative_scores = np.zeros(len(books_features))
        
        neighbor_book_ids = eval_book_ids.cat.categories[
            neighbor_indices
        ]
        
        for neighbor_book_id, score in zip(
            neighbor_book_ids,
            neighbor_scores
        ):
            
            if neighbor_book_id in book_id_to_index:
                row_index = book_id_to_index[neighbor_book_id]
                collaborative_scores[row_index] = score
        
        # Hybrid score
        hybrid_scores = (
            alpha * content_scores
            + (1 - alpha) * collaborative_scores
        )
        
        hybrid_scores[content_idx] = -1
        
        top_indices = np.argsort(
            hybrid_scores
        )[::-1][:10]
        
        recommended_ids = books_features.iloc[
            top_indices
        ]["book_id"].values
        
        if test_book in recommended_ids:
            hits += 1
    
    hit_rate = hits / len(evaluation_sample)
    precision = hits / (len(evaluation_sample) * 10)
    recall = hit_rate
    
    experiment_metrics.append({
        "alpha": alpha,
        "content_weight": alpha,
        "collaborative_weight": 1 - alpha,
        "precision_at_10": precision,
        "recall_at_10": recall,
        "hit_rate_at_10": hit_rate
    })

experiment_metrics_df = pd.DataFrame(experiment_metrics)

experiment_metrics_df

,alpha,content_weight,collaborative_weight,precision_at_10,recall_at_10,hit_rate_at_10
0,0.5,0.5,0.5,0.0039,0.039,0.039
1,0.6,0.6,0.4,0.0036,0.036,0.036
2,0.7,0.7,0.3,0.0032,0.032,0.032
3,0.8,0.8,0.2,0.0029,0.029,0.029
4,0.4,0.4,0.6,0.0040,0.040,0.040
5,0.3,0.3,0.7,0.0042,0.042,0.042


In [51]:
# Evaluate hybrid recommender at K = 5

alpha_values = [0.5, 0.6, 0.7, 0.8, 0.4, 0.3]

results_at_5 = []

for alpha in alpha_values:

    hits = 0

    for _, row in evaluation_sample.iterrows():

        seed_book_id = row["seed_book_id"]
        test_book_id = row["test_book_id"]

        if seed_book_id not in book_id_to_index:
            continue

        # Content scores
        content_idx = book_id_to_index[seed_book_id]
        content_scores = content_similarity[content_idx]

        # Collaborative scores
        collaborative_scores = np.zeros(len(books_features))

        if seed_book_id in collaborative_recommendation_map:
            similar_book_ids = collaborative_recommendation_map[seed_book_id][:5]

            # Give collaborative neighbours a score
            for rank, similar_book_id in enumerate(similar_book_ids):
                if similar_book_id in book_id_to_index:
                    idx = book_id_to_index[similar_book_id]
                    collaborative_scores[idx] = 1 - (rank / 5)

        # Hybrid score
        hybrid_scores = (
            alpha * content_scores
            + (1 - alpha) * collaborative_scores
        )

        # Remove seed book
        hybrid_scores[content_idx] = -1

        # Top 5
        top_indices = np.argsort(hybrid_scores)[::-1][:5]

        recommended_books = books_features.iloc[
            top_indices
        ]["book_id"].tolist()

        if test_book_id in recommended_books:
            hits += 1

    hit_rate = hits / len(evaluation_sample)

    results_at_5.append({
        "alpha": alpha,
        "content_weight": alpha,
        "collaborative_weight": 1 - alpha,
        "hit_rate_at_5": hit_rate
    })

results_at_5 = pd.DataFrame(results_at_5)

results_at_5

,alpha,content_weight,collaborative_weight,hit_rate_at_5
0,0.5,0.5,0.5,0.022
1,0.6,0.6,0.4,0.021
2,0.7,0.7,0.3,0.019
3,0.8,0.8,0.2,0.019
4,0.4,0.4,0.6,0.022
5,0.3,0.3,0.7,0.022


In [52]:
# Evaluate hybrid recommender at K = 20

alpha_values = [0.5, 0.6, 0.7, 0.8, 0.4, 0.3]

results_at_20 = []

for alpha in alpha_values:

    hits = 0

    for _, row in evaluation_sample.iterrows():

        seed_book_id = row["seed_book_id"]
        test_book_id = row["test_book_id"]

        if seed_book_id not in book_id_to_index:
            continue

        # Content scores
        content_idx = book_id_to_index[seed_book_id]
        content_scores = content_similarity[content_idx]

        # Collaborative scores
        collaborative_scores = np.zeros(len(books_features))

        if seed_book_id in collaborative_recommendation_map:
            similar_book_ids = collaborative_recommendation_map[seed_book_id][:20]

            for rank, similar_book_id in enumerate(similar_book_ids):
                if similar_book_id in book_id_to_index:
                    idx = book_id_to_index[similar_book_id]
                    collaborative_scores[idx] = 1 - (rank / 20)

        # Hybrid score
        hybrid_scores = (
            alpha * content_scores
            + (1 - alpha) * collaborative_scores
        )

        # Remove seed book
        hybrid_scores[content_idx] = -1

        # Top 20
        top_indices = np.argsort(hybrid_scores)[::-1][:20]

        recommended_books = books_features.iloc[
            top_indices
        ]["book_id"].tolist()

        if test_book_id in recommended_books:
            hits += 1

    hit_rate = hits / len(evaluation_sample)

    results_at_20.append({
        "alpha": alpha,
        "content_weight": alpha,
        "collaborative_weight": 1 - alpha,
        "hit_rate_at_20": hit_rate
    })

results_at_20 = pd.DataFrame(results_at_20)

results_at_20

,alpha,content_weight,collaborative_weight,hit_rate_at_20
0,0.5,0.5,0.5,0.052
1,0.6,0.6,0.4,0.053
2,0.7,0.7,0.3,0.050
3,0.8,0.8,0.2,0.048
4,0.4,0.4,0.6,0.052
5,0.3,0.3,0.7,0.052


In [53]:
# Evaluate hybrid recommender at K = 20

alpha_values = [0.5, 0.6, 0.7, 0.8, 0.4, 0.3]

results_at_20 = []

for alpha in alpha_values:

    hits = 0

    for _, row in evaluation_sample.iterrows():

        seed_book_id = row["seed_book_id"]
        test_book_id = row["test_book_id"]

        if seed_book_id not in book_id_to_index:
            continue

        # Content scores
        content_idx = book_id_to_index[seed_book_id]
        content_scores = content_similarity[content_idx]

        # Collaborative scores
        collaborative_scores = np.zeros(len(books_features))

        if seed_book_id in collaborative_recommendation_map:
            similar_book_ids = collaborative_recommendation_map[seed_book_id][:20]

            for rank, similar_book_id in enumerate(similar_book_ids):
                if similar_book_id in book_id_to_index:
                    idx = book_id_to_index[similar_book_id]
                    collaborative_scores[idx] = 1 - (rank / 20)

        # Hybrid score
        hybrid_scores = (
            alpha * content_scores
            + (1 - alpha) * collaborative_scores
        )

        # Remove seed book
        hybrid_scores[content_idx] = -1

        # Top 20
        top_indices = np.argsort(hybrid_scores)[::-1][:20]

        recommended_books = books_features.iloc[
            top_indices
        ]["book_id"].tolist()

        if test_book_id in recommended_books:
            hits += 1

    hit_rate = hits / len(evaluation_sample)

    results_at_20.append({
        "alpha": alpha,
        "content_weight": alpha,
        "collaborative_weight": 1 - alpha,
        "hit_rate_at_20": hit_rate
    })

results_at_20 = pd.DataFrame(results_at_20)

results_at_20

,alpha,content_weight,collaborative_weight,hit_rate_at_20
0,0.5,0.5,0.5,0.052
1,0.6,0.6,0.4,0.053
2,0.7,0.7,0.3,0.050
3,0.8,0.8,0.2,0.048
4,0.4,0.4,0.6,0.052
5,0.3,0.3,0.7,0.052


In [54]:
# Precompute collaborative similarity scores for each evaluation seed

collaborative_score_map = {}

for i, book_id in enumerate(unique_seed_books):

    neighbor_indices = indices[i][1:]
    neighbor_distances = distances[i][1:]

    # Convert cosine distance to cosine similarity
    neighbor_similarities = 1 - neighbor_distances

    neighbor_book_ids = eval_book_ids.cat.categories[
        neighbor_indices
    ].tolist()

    collaborative_score_map[book_id] = dict(
        zip(
            neighbor_book_ids,
            neighbor_similarities
        )
    )

print("Collaborative score map created.")
print("Seed books:", len(collaborative_score_map))

# Check one example
sample_seed = unique_seed_books[0]

list(collaborative_score_map[sample_seed].items())[:5]

Collaborative score map created.
Seed books: 813


[(1290, np.float64(0.7560581684776673)),
 (1903, np.float64(0.5567458417454207)),
 (36, np.float64(0.27841348202986826)),
 (241, np.float64(0.17673245242145563)),
 (17, np.float64(0.16617206210358038))]

In [55]:
# Correct hybrid evaluation using actual cosine similarities

alpha_values = [0.5, 0.6, 0.7, 0.8, 0.4, 0.3]
k_values = [5, 10, 20]

evaluation_results = []

for k in k_values:

    for alpha in alpha_values:

        hits = 0

        for _, row in evaluation_sample.iterrows():

            seed_book_id = row["seed_book_id"]
            test_book_id = row["test_book_id"]

            if seed_book_id not in book_id_to_index:
                continue

            # -----------------------------
            # CONTENT SCORES
            # -----------------------------

            content_idx = book_id_to_index[seed_book_id]

            content_scores = content_similarity[
                content_idx
            ].copy()

            # -----------------------------
            # COLLABORATIVE SCORES
            # -----------------------------

            collaborative_scores = np.zeros(
                len(books_features)
            )

            if seed_book_id in collaborative_score_map:

                for similar_book_id, score in collaborative_score_map[
                    seed_book_id
                ].items():

                    if similar_book_id in book_id_to_index:

                        idx = book_id_to_index[
                            similar_book_id
                        ]

                        collaborative_scores[idx] = score

            # -----------------------------
            # HYBRID SCORE
            # -----------------------------

            hybrid_scores = (
                alpha * content_scores
                + (1 - alpha) * collaborative_scores
            )

            # Remove seed book
            hybrid_scores[content_idx] = -1

            # -----------------------------
            # TOP K
            # -----------------------------

            top_indices = np.argsort(
                hybrid_scores
            )[::-1][:k]

            recommended_books = books_features.iloc[
                top_indices
            ]["book_id"].tolist()

            # -----------------------------
            # HIT
            # -----------------------------

            if test_book_id in recommended_books:
                hits += 1

        hit_rate = hits / len(evaluation_sample)

        evaluation_results.append({
            "alpha": alpha,
            "content_weight": alpha,
            "collaborative_weight": 1 - alpha,
            "k": k,
            "hit_rate": hit_rate
        })

evaluation_results = pd.DataFrame(
    evaluation_results
)

evaluation_results

,alpha,content_weight,collaborative_weight,k,hit_rate
0,0.5,0.5,0.5,5,0.022
1,0.6,0.6,0.4,5,0.019
2,0.7,0.7,0.3,5,0.018
3,0.8,0.8,0.2,5,0.018
4,0.4,0.4,0.6,5,0.022
5,0.3,0.3,0.7,5,0.023
6,0.5,0.5,0.5,10,0.039
7,0.6,0.6,0.4,10,0.036
8,0.7,0.7,0.3,10,0.032
9,0.8,0.8,0.2,10,0.029


In [56]:
# Evaluate pure content and pure collaborative baselines

baseline_results = []

for k in [5, 10, 20]:

    content_hits = 0
    collaborative_hits = 0

    for _, row in evaluation_sample.iterrows():

        seed_book_id = row["seed_book_id"]
        test_book_id = row["test_book_id"]

        # -----------------------------
        # CONTENT-BASED
        # -----------------------------

        content_recommendations = get_content_recommendations(
            seed_book_id,
            k=k
        )

        if test_book_id in content_recommendations:
            content_hits += 1

        # -----------------------------
        # COLLABORATIVE
        # -----------------------------

        if seed_book_id in collaborative_recommendation_map:

            collaborative_recommendations = (
                collaborative_recommendation_map[seed_book_id][:k]
            )

            if test_book_id in collaborative_recommendations:
                collaborative_hits += 1

    baseline_results.append({
        "k": k,
        "content_hit_rate": content_hits / len(evaluation_sample),
        "collaborative_hit_rate": collaborative_hits / len(evaluation_sample)
    })

baseline_results = pd.DataFrame(baseline_results)

baseline_results

,k,content_hit_rate,collaborative_hit_rate
0,5,0.019,0.023
1,10,0.025,0.043
2,20,0.042,0.043


In [57]:
# Calculate Precision@K and Recall@K
# Each evaluation case has exactly one relevant hidden book.

complete_metrics = evaluation_results.copy()

complete_metrics["precision_at_k"] = (
    complete_metrics["hit_rate"] / complete_metrics["k"]
)

complete_metrics["recall_at_k"] = (
    complete_metrics["hit_rate"]
)

complete_metrics = complete_metrics[
    [
        "alpha",
        "content_weight",
        "collaborative_weight",
        "k",
        "precision_at_k",
        "recall_at_k",
        "hit_rate"
    ]
]

complete_metrics

,alpha,content_weight,collaborative_weight,k,precision_at_k,recall_at_k,hit_rate
0,0.5,0.5,0.5,5,0.00440,0.022,0.022
1,0.6,0.6,0.4,5,0.00380,0.019,0.019
2,0.7,0.7,0.3,5,0.00360,0.018,0.018
3,0.8,0.8,0.2,5,0.00360,0.018,0.018
4,0.4,0.4,0.6,5,0.00440,0.022,0.022
5,0.3,0.3,0.7,5,0.00460,0.023,0.023
6,0.5,0.5,0.5,10,0.00390,0.039,0.039
7,0.6,0.6,0.4,10,0.00360,0.036,0.036
8,0.7,0.7,0.3,10,0.00320,0.032,0.032
9,0.8,0.8,0.2,10,0.00290,0.029,0.029


The collaborative model provides the strongest baseline performance at K=10, while the hybrid model with greater collaborative weighting performs competitively and achieves higher Hit Rate@20. Among the tested hybrid configurations, α=0.3 is the strongest overall choice, with α=0.4 performing equally well at K=20.

In [58]:
# Inspect the current hybrid experiment log

hybrid_experiments = pd.read_csv(
    "../experiments/hybrid_experiments.csv"
)

hybrid_experiments

,experiment,content_weight,collaborative_weight,precision_at_10,recall_at_10,hit_rate_at_10,notes
0,EXP-01,0.5,0.5,NaN,NaN,NaN,Baseline -> 3.9% Hit@10
1,EXP-02,0.6,0.4,NaN,NaN,NaN,Current configuration
2,EXP-03,0.7,0.3,NaN,NaN,NaN,Content-heavy
3,EXP-04,0.8,0.2,NaN,NaN,NaN,Strong content
4,EXP-05,0.4,0.6,NaN,NaN,NaN,Collaborative-heavy
5,EXP-06,0.3,0.7,NaN,NaN,NaN,Strong collaborative


In [60]:
# Update hybrid experiment log with confirmed @10 results

hybrid_experiments["precision_at_10"] = [
    0.0039,
    0.0036,
    0.0032,
    0.0029,
    0.0040,
    0.0042
]

hybrid_experiments["recall_at_10"] = [
    0.039,
    0.036,
    0.032,
    0.029,
    0.040,
    0.042
]

hybrid_experiments["hit_rate_at_10"] = [
    0.039,
    0.036,
    0.032,
    0.029,
    0.040,
    0.042
]

hybrid_experiments["notes"] = [
    "Baseline",
    "Current configuration",
    "Content-heavy",
    "Strong content",
    "Collaborative-heavy",
    "Strong collaborative"
]

hybrid_experiments.to_csv(
    "../experiments/hybrid_experiments.csv",
    index=False
)

hybrid_experiments

,experiment,content_weight,collaborative_weight,precision_at_10,recall_at_10,hit_rate_at_10,notes
0,EXP-01,0.5,0.5,0.0039,0.039,0.039,Baseline
1,EXP-02,0.6,0.4,0.0036,0.036,0.036,Current configuration
2,EXP-03,0.7,0.3,0.0032,0.032,0.032,Content-heavy
3,EXP-04,0.8,0.2,0.0029,0.029,0.029,Strong content
4,EXP-05,0.4,0.6,0.0040,0.040,0.040,Collaborative-heavy
5,EXP-06,0.3,0.7,0.0042,0.042,0.042,Strong collaborative
